# Task 6: обработка benchmark CSV

Ноутбук загружает CSV-файлы из `results/`, срезает выбросы по квантилям внутри каждой группы `mode/device/size`, усредняет результаты, строит таблицы и графики для отчета.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path('results')
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CSV_GLOB = 'benchmark*.csv'
LOW_Q = 0.05
HIGH_Q = 0.95

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

## 1. Загрузка CSV

In [ ]:
def infer_device_from_name(path: Path) -> str:
    name = path.stem.lower()
    if 'gpu' in name:
        return 'gpu'
    if 'multicore' in name:
        return 'cpu-multicore'
    if 'serial' in name or 'host' in name or 'onecore' in name:
        return 'cpu-onecore'
    return 'unknown'

def normalize_frame(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if 'mode' not in df.columns:
        df['mode'] = 'openacc' if 'device' in df.columns else 'unknown'
    if 'device' not in df.columns:
        df['device'] = infer_device_from_name(path)
    if 'time_sec' not in df.columns and 'time' in df.columns:
        df = df.rename(columns={'time': 'time_sec'})
    df['source_file'] = path.name
    df['mode'] = df['mode'].astype(str).str.lower()
    df['device'] = df['device'].astype(str).str.lower()
    return df

paths = sorted(RESULTS_DIR.glob(CSV_GLOB))
if not paths:
    raise FileNotFoundError(f'CSV files not found: {RESULTS_DIR / CSV_GLOB}')

raw = pd.concat([normalize_frame(p) for p in paths], ignore_index=True)
raw['impl'] = np.where(raw['mode'].eq('serial'), 'serial/cpu-onecore', raw['mode'] + '/' + raw['device'])
raw = raw.sort_values(['impl', 'size', 'run']).reset_index(drop=True)

print(f'Loaded {len(raw)} rows from {len(paths)} files')
display(raw.head())
display(raw.groupby(['impl', 'size']).size().rename('runs').reset_index())

## 2. Срез выбросов по квантилям

In [ ]:
def trim_by_quantiles(df: pd.DataFrame, low=LOW_Q, high=HIGH_Q, value='time_sec') -> pd.DataFrame:
    def trim_group(group):
        if len(group) < 4:
            return group.copy()
        lo = group[value].quantile(low)
        hi = group[value].quantile(high)
        return group[(group[value] >= lo) & (group[value] <= hi)].copy()
    return df.groupby(['impl', 'size'], group_keys=False).apply(trim_group)

clean = trim_by_quantiles(raw)
removed = len(raw) - len(clean)
print(f'Removed {removed} rows of {len(raw)} ({removed / len(raw) * 100:.1f}%)')
display(clean.groupby(['impl', 'size']).size().rename('runs_after_trim').reset_index())

## 3. Усредненные таблицы

In [ ]:
summary = (
    clean.groupby(['impl', 'mode', 'device', 'size'], as_index=False)
    .agg(
        runs=('time_sec', 'count'),
        time_mean=('time_sec', 'mean'),
        time_median=('time_sec', 'median'),
        time_std=('time_sec', 'std'),
        iterations_mean=('iterations', 'mean'),
        error_mean=('error', 'mean'),
    )
)
summary['time_std'] = summary['time_std'].fillna(0.0)
summary['size_label'] = summary['size'].astype(str) + 'x' + summary['size'].astype(str)
summary = summary.sort_values(['impl', 'size']).reset_index(drop=True)

summary_path = RESULTS_DIR / 'summary_trimmed.csv'
summary.to_csv(summary_path, index=False)
print(f'Saved {summary_path}')
display(summary)

In [ ]:
report_table = summary[['impl', 'size_label', 'runs', 'time_mean', 'time_std', 'iterations_mean', 'error_mean']].copy()
report_table.columns = ['Implementation', 'Grid', 'Runs', 'Mean time, s', 'Std, s', 'Iterations', 'Error']
report_table['Mean time, s'] = report_table['Mean time, s'].map('{:.6f}'.format)
report_table['Std, s'] = report_table['Std, s'].map('{:.6f}'.format)
report_table['Iterations'] = report_table['Iterations'].map('{:.0f}'.format)
report_table['Error'] = report_table['Error'].map('{:.3e}'.format)
display(report_table)
report_table.to_csv(RESULTS_DIR / 'report_table.csv', index=False)

## 4. Ускорение относительно baseline

In [ ]:
def baseline_impl(summary_df):
    candidates = [x for x in summary_df['impl'].unique() if x.startswith('serial/')]
    return candidates[0] if candidates else None

base_name = baseline_impl(summary)
if base_name is None:
    print('Serial baseline not found; speedup table is skipped.')
    speedup_table = pd.DataFrame()
else:
    base = summary[summary['impl'].eq(base_name)].set_index('size')['time_mean']
    rows = []
    for _, row in summary.iterrows():
        if row['size'] in base.index:
            rows.append({
                'impl': row['impl'],
                'size': row['size'],
                'size_label': row['size_label'],
                'time_mean': row['time_mean'],
                'speedup_vs_serial': base.loc[row['size']] / row['time_mean'],
            })
    speedup_table = pd.DataFrame(rows)
    speedup_table.to_csv(RESULTS_DIR / 'speedup_table.csv', index=False)
    display(speedup_table)

## 5. Графики

In [ ]:
def savefig(name):
    path = PLOTS_DIR / name
    plt.tight_layout()
    plt.savefig(path)
    print(f'Saved {path}')

sizes = sorted(summary['size'].unique())
impls = list(summary['impl'].unique())
x = np.arange(len(sizes))
width = min(0.8 / max(len(impls), 1), 0.25)

fig, ax = plt.subplots(figsize=(9, 4.8))
for idx, impl in enumerate(impls):
    data = summary[summary['impl'].eq(impl)].set_index('size').reindex(sizes)
    offset = (idx - (len(impls) - 1) / 2) * width
    ax.bar(x + offset, data['time_mean'], width, yerr=data['time_std'], capsize=3, label=impl)
ax.set_xticks(x)
ax.set_xticklabels([f'{s}x{s}' for s in sizes])
ax.set_xlabel('Grid size')
ax.set_ylabel('Mean time, s')
ax.set_title('Execution time by implementation')
ax.legend()
savefig('time_grouped_bar.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for impl in impls:
    data = summary[summary['impl'].eq(impl)].sort_values('size')
    ax.plot(data['size'], data['time_mean'], marker='o', linewidth=2, label=impl)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Grid size N for NxN')
ax.set_ylabel('Mean time, s')
ax.set_title('Execution time scaling')
ax.legend()
savefig('time_scaling_log.png')
plt.show()

In [ ]:
if not speedup_table.empty:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for impl in speedup_table['impl'].unique():
        if impl == base_name:
            continue
        data = speedup_table[speedup_table['impl'].eq(impl)].sort_values('size')
        ax.plot(data['size'], data['speedup_vs_serial'], marker='o', linewidth=2, label=impl)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('Grid size N for NxN')
    ax.set_ylabel('Speedup vs serial CPU')
    ax.set_title('Acceleration relative to baseline')
    ax.legend()
    savefig('speedup_vs_serial.png')
    plt.show()

In [ ]:
pivot = summary.pivot_table(index='size_label', columns='impl', values='time_mean', aggfunc='mean')
display(pivot)

fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(pivot.columns)), 3.8))
im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        value = pivot.values[i, j]
        if np.isfinite(value):
            ax.text(j, i, f'{value:.3g}', ha='center', va='center', color='white')
ax.set_title('Mean time heatmap, s')
fig.colorbar(im, ax=ax, label='Seconds')
savefig('time_heatmap.png')
plt.show()

## 6. Просмотр сохраненных матриц

In [ ]:
grid_files = sorted(RESULTS_DIR.glob('grid*.csv')) + sorted(RESULTS_DIR.glob('grid*.txt'))
for path in grid_files:
    try:
        mat = np.loadtxt(path, delimiter=',')
    except ValueError:
        mat = np.loadtxt(path)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    image = ax.imshow(mat, cmap='inferno', origin='upper')
    ax.set_title(path.name)
    fig.colorbar(image, ax=ax, label='Temperature')
    savefig(path.stem + '_matrix.png')
    plt.show()
    display(pd.DataFrame(mat).round(3))